# 🚀 AirLLM on Google Colab: 超大模型極限推論完整實戰指南

> **專案用途**：[AirLLM](https://github.com/Child-pi/airllm) (由 Gavin Li 開發) 透過**分層串流 (Layer-by-layer Streaming)** 技術，在單張低顯存消費級 GPU (如 4GB / 8GB / 16GB T4) 上運行 70B 乃至 405B、671B (DeepSeek-V3) 超大型語言模型。
>
> 💡 **核心哲學**：以硬碟 I/O 換取顯存空間。Transformer 在推論時，每個時間點只需要計算其中一層 Decoder。AirLLM 將權重切片存於磁碟，逐層載入 GPU 計算後釋放，從而突破硬體顯存極限。
>
> 🛡️ **本 Notebook 融入之 Antigravity Colab 實戰加固技術**：
> - ⚡ Google Drive 冪等性掛載防呆（零等待零重複彈窗）
> - 📂 工作區目錄獨立命名空間隔離 (`isolate-project-workspace`)
> - 🛡️ Linux OOM-Killer 與顯存碎片防護 (`expandable_segments:True`)
> - 💾 本機磁碟空間健康監控（防止大型模型撐爆 Colab 虛擬機）
> - 🔄 背景守護進程解耦推論 (Nohup Daemon)，避免網頁或連線中斷丟失進度


## ⚙️ Step 1: 硬體環境診斷與顯存防護配置
檢查目前分配到的 GPU 型號、顯存容量以及 Colab 虛擬機磁碟剩餘空間。


In [ ]:
# @title 1. 硬體診斷與環境變數注入
import os
import shutil
import torch

# 1. 啟用 PyTorch 顯存可擴展段分配，預防顯存碎片引發 OOM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONUNBUFFERED"] = "1"

# 2. GPU 狀態檢查
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ 偵測到 GPU: {gpu_name}")
    print(f"📊 總顯存 (VRAM): {gpu_mem:.2f} GB")
else:
    print("⚠️ 未檢測到 GPU！請點選上方選單「執行階段」->「變更執行階段類型」-> 選擇「T4 GPU」或更高階 GPU。")

# 3. 本機硬碟空間檢查 (AirLLM 需要空間存放分層權重)
total, used, free = shutil.disk_usage("/content")
free_gb = free / (1024**3)
total_gb = total / (1024**3)
print(f"💾 本機 /content 剩餘磁碟空間: {free_gb:.2f} GB / 總共 {total_gb:.2f} GB")

if free_gb < 20:
    print("⚠️ 警告：剩餘磁碟空間小於 20GB，若下載 70B 未量化權重可能會磁碟不足，建議使用 4-bit 量化或掛載 Google Drive！")
else:
    print("✅ 磁碟空間充足，適合進行模型分層串流推論。")


## ☁️ Step 2: Google Drive 冪等掛載與持久化快取 (選用)
若希望將模型分層切片 (`layer_shards`) 保存於雲端硬碟，避免每次重新連接 Colab 都需要重新下載與切片數十 GB 模型，可啟用 Google Drive。


In [ ]:
# @title 2. 掛載 Google Drive (冪等檢查，防重複彈窗卡死)
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/AirLLM_Workspace"

if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        print("🔗 正在掛載 Google Drive...")
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print("⚡ Google Drive 已經掛載，跳過 OAuth 請求！")
    
    os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
    os.makedirs(os.path.join(DRIVE_WORKSPACE, "shards"), exist_ok=True)
    print(f"📁 專屬持久化工作區已就緒: {DRIVE_WORKSPACE}")
else:
    print("ℹ️ 使用 Colab 本機暫存空間 (/content/airllm_cache)")
    os.makedirs("/content/airllm_cache", exist_ok=True)


## 📦 Step 3: 安裝 AirLLM 與核心依賴
安裝最新版 `airllm` 以及 `bitsandbytes` (提供 4-bit / 8-bit 量化支援)、`accelerate`、`transformers`。


In [ ]:
# @title 3. 安裝依賴套件
!pip install -q -U airllm bitsandbytes accelerate transformers sentencepiece safetensors huggingface_hub

print("✅ AirLLM 與相依套件安裝完成！")


## 🧠 Step 4: 模型與推論參數設定
AirLLM 支援 `AutoModel` 自動識別架構。
對於 Google Colab（尤其是免費 T4 GPU，磁碟約 70~100GB）：
- **推薦測試**：`Qwen/Qwen2.5-7B-Instruct` 或 `Qwen/Qwen2.5-14B-Instruct`（下載與切片速度快）
- **大模型突破**：`garage-bAInd/Platypus2-70B-instruct` 或 `meta-llama/Meta-Llama-3-70B-Instruct`（需注意磁碟空間，建議選 `4bit` 壓縮）


In [ ]:
# @title 4. 設定模型參數
# @markdown ### 模型名稱或 Hugging Face ID
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" # @param ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct", "garage-bAInd/Platypus2-70B-instruct", "meta-llama/Meta-Llama-3-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.2"] {allow-input: true}

# @markdown ### 壓縮模式 (4bit 可提升 3 倍載入速度並節省磁碟)
COMPRESSION = "4bit" # @param ["none", "4bit", "8bit"]

# @markdown ### 生成設定
MAX_INPUT_LENGTH = 256 # @param {type:"integer"}
MAX_NEW_TOKENS = 64 # @param {type:"integer"}

# @markdown ### HuggingFace Token (可留空，若使用 Llama-3 等受保護模型則需填寫)
HF_TOKEN = "" # @param {type:"string"}

# 決定切片暫存路徑
if USE_GOOGLE_DRIVE and os.path.exists(DRIVE_WORKSPACE):
    # 以模型名稱建立專屬目錄
    safe_name = MODEL_ID.replace("/", "_")
    SHARDS_PATH = os.path.join(DRIVE_WORKSPACE, "shards", safe_name)
else:
    SHARDS_PATH = "/content/airllm_cache/shards"

os.makedirs(SHARDS_PATH, exist_ok=True)
compression_arg = None if COMPRESSION == "none" else COMPRESSION

print(f"📌 模型: {MODEL_ID}")
print(f"📌 壓縮模式: {compression_arg}")
print(f"📌 分層切片存放路徑: {SHARDS_PATH}")


## 💬 Step 5: 前景直接推論 (即時互動測試)
直接在 Cell 中初始化 AirLLM，並對輸入問題生成回應。
> ⏱️ **提醒**：第一次執行時需要從 Hugging Face 下載模型並切片存入磁碟，需要花費數分鐘；切片完成後，後續每次推論即可直接從磁碟串流讀取！


In [ ]:
# @title 5. 載入模型並執行推論
import time
from airllm import AutoModel

print("⏳ 正在初始化 AirLLM 模型 (首次載入需進行權重下載與分層切片)...")
start_init = time.time()

kwargs = {
    "layer_shards_saving_path": SHARDS_PATH
}
if compression_arg:
    kwargs["compression"] = compression_arg
if HF_TOKEN.strip():
    kwargs["hf_token"] = HF_TOKEN.strip()

model = AutoModel.from_pretrained(MODEL_ID, **kwargs)
init_duration = time.time() - start_init
print(f"✅ 模型就緒！初始化耗時: {init_duration:.1f} 秒\n")

# 測試 Prompt
PROMPT_TEXT = "請用繁體中文以三點簡要說明人工智慧（AI）未來的發展趨勢。" # @param {type:"string"}

print(f"❓ 輸入 Prompt: {PROMPT_TEXT}\n")
print("🚀 開始串流推論 (分層依序計算中)...")

infer_start = time.time()
input_tokens = model.tokenizer(
    [PROMPT_TEXT],
    return_tensors="pt",
    return_attention_mask=True,
    truncation=True,
    max_length=MAX_INPUT_LENGTH,
    padding=True
)

output = model.generate(
    input_tokens['input_ids'],
    max_new_tokens=MAX_NEW_TOKENS,
    use_cache=True
)

infer_duration = time.time() - infer_start
response = model.tokenizer.batch_decode(output, skip_special_tokens=True)

print("\n🎉 ========== 生成結果 ==========")
print(response[0])
print("================================")
print(f"⏱️ 推論耗時: {infer_duration:.2f} 秒 (約 {MAX_NEW_TOKENS/infer_duration:.2f} tokens/sec)")
if torch.cuda.is_available():
    print(f"🔥 GPU 峰值顯存佔用: {torch.cuda.max_memory_allocated() / (1024**2):.2f} MB (極限輕量！)")


## 🛡️ Step 6: 背景守護推論模式 (解耦防網頁斷線)
當您需要使用 70B 模型生成長篇文檔或執行批次處理任務時，生成時間可能達數十分鐘。
此時若依賴瀏覽器前景 Cell，遇到筆電休眠或 Wi-Fi 抖動時 Cell 就會被中止。
**解方**：將任務寫成獨立腳本，使用 `nohup` 移至背景執行，並即時串流觀看日誌。


In [ ]:
# @title 6.1 建立背景推論任務腳本
script_content = f'''import os
import time
import torch
from airllm import AutoModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONUNBUFFERED"] = "1"

MODEL_ID = "{MODEL_ID}"
SHARDS_PATH = "{SHARDS_PATH}"
COMPRESSION = "{compression_arg}"
HF_TOKEN = "{HF_TOKEN.strip()}"

print("[BG Task] 啟動背景推論腳本...")
start_time = time.time()

kwargs = {{"layer_shards_saving_path": SHARDS_PATH}}
if COMPRESSION != "None":
    kwargs["compression"] = COMPRESSION
if HF_TOKEN:
    kwargs["hf_token"] = HF_TOKEN

model = AutoModel.from_pretrained(MODEL_ID, **kwargs)
print(f"[BG Task] 模型載入完成，耗時: {{time.time() - start_time:.2f}}s")

prompts = [
    "請為即將面試 AI 工程師的候選人準備 3 個核心技術考題與參考答案。"
]

input_tokens = model.tokenizer(
    prompts,
    return_tensors="pt",
    return_attention_mask=True,
    truncation=True,
    max_length=256,
    padding=True
)

print("[BG Task] 開始生成...")
gen_start = time.time()
output = model.generate(input_tokens["input_ids"], max_new_tokens=128, use_cache=True)
gen_time = time.time() - gen_start

res = model.tokenizer.batch_decode(output, skip_special_tokens=True)
print("\n[BG Task] ===== 背景推論完成結果 =====")
print(res[0])
print(f"[BG Task] 生成耗時: {{gen_time:.2f}}s")
'''

with open("/content/bg_airllm_infer.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("✅ 背景任務腳本 /content/bg_airllm_infer.py 已生成！")


In [ ]:
# @title 6.2 啟動背景守護進程 (Nohup Daemon)
import subprocess

# 檢查是否已在運行
status_code = subprocess.call(["pgrep", "-f", "bg_airllm_infer.py"])
if status_code == 0:
    print("⚠️ 背景推論任務已在執行中！")
else:
    cmd = "PYTHONUNBUFFERED=1 nohup python /content/bg_airllm_infer.py > /content/airllm_task.log 2>&1 & echo $! > /content/airllm_task.pid"
    os.system(cmd)
    pid = open("/content/airllm_task.pid").read().strip()
    print(f"🚀 背景推論任務已成功啟動！PID: {pid}")
    print("📄 日誌輸出至: /content/airllm_task.log")
    print("💡 您可以隨意切換分頁，任務持續在背景執行！")


In [ ]:
# @title 6.3 查看即時背景日誌 (可反覆點擊執行更新)
import time
from IPython.display import clear_output

if os.path.exists("/content/airllm_task.log"):
    with open("/content/airllm_task.log", "r", encoding="utf-8", errors="ignore") as f:
        log_content = f.read()
    
    # 檢查進程存活
    is_running = (subprocess.call(["pgrep", "-f", "bg_airllm_infer.py"]) == 0)
    status_str = "🟢 正在運行中" if is_running else "⚪ 任務已完成或已終止"
    
    print(f"=== 任務狀態: {status_str} ===")
    print(log_content[-1500:] if len(log_content) > 1500 else log_content)
else:
    print("ℹ️ 尚未找到日誌檔，請先執行 6.2 啟動背景任務。")


## 🧹 Step 7: 磁碟空間釋放與資源管理
AirLLM 會在 `layer_shards_saving_path` 存放切片檔案。若測試完畢需要清空 Colab 空間，可使用以下儲存格進行安全清理。


In [ ]:
# @title 7. 檢查與清理分層切片快取
CONFIRM_DELETE = False # @param {type:"boolean"}
TARGET_PATH_TO_CLEAN = "/content/airllm_cache" # @param ["/content/airllm_cache", "/content/drive/MyDrive/AirLLM_Workspace/shards"] {allow-input: true}

def get_dir_size(path):
    total = 0
    if not os.path.exists(path):
        return 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024**3)

size_gb = get_dir_size(TARGET_PATH_TO_CLEAN)
print(f"📊 目標目錄: {TARGET_PATH_TO_CLEAN}")
print(f"📦 目前佔用磁碟大小: {size_gb:.2f} GB")

if CONFIRM_DELETE:
    shutil.rmtree(TARGET_PATH_TO_CLEAN, ignore_errors=True)
    os.makedirs(TARGET_PATH_TO_CLEAN, exist_ok=True)
    print(f"✅ 已成功清理: {TARGET_PATH_TO_CLEAN}")
else:
    print("ℹ️ 未勾選 CONFIRM_DELETE，保留快取檔案。")
